# IOAI — 2026 Local Onia Museums (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/muzee.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-museums.zip', '/tmp/muzee.zip')
    with zipfile.ZipFile('/tmp/muzee.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/muzee.csv' if os.path.exists('data/muzee.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
seed = 42

root_path = "data"


# Data

In [ ]:
muzee = pd.read_csv(f"{root_path}/muzee.csv")

muzee.head()

# Subtask 1

In [ ]:
subtask1 = len(muzee["denumirea (română)"])

subtask1

# Subtask 2

In [ ]:
subtask2 = len(muzee[muzee["județul"] == "București"])

subtask2

# Subtask 3

In [ ]:
len(muzee.columns)

In [ ]:
subtask3 = (muzee.isna().sum() > 0).sum()

subtask3

# Subtask 4

In [ ]:
from copy import deepcopy

df = deepcopy(muzee)

df["an_fixed"] = df["anul înființării"].apply(
    lambda x: x.split(",")[0].split(";")[0] if type(x) == str else x
)

In [ ]:
df.groupby("an_fixed")["codul entității muzeale"].nunique().sort_values(ascending=False)

In [ ]:
subtask4 = 2006

# Subtask 5

In [ ]:
subtask5 = (
    muzee.groupby("județul")["codul entității muzeale"]
    .nunique()
    .sort_values(ascending=False)
)

subtask5

# Subtask 6

In [ ]:
muzee.shape, len(muzee.columns), muzee.columns.size

In [ ]:
subtask6 = round((muzee.notna().sum(axis=1)) / muzee.shape[1] * 100, 2)
subtask6

In [ ]:
muzee.shape

# Subtask 7

In [ ]:
subtask7 = subtask6.mean()

subtask7

# Subtask 8

In [ ]:
subtask8 = (subtask6 == subtask6.max()).sum() / len(muzee) * 100

subtask8 = float(subtask8.item())
subtask8

# Submission

In [ ]:
def build_subtask(id, sid, ans):
    return pd.DataFrame({"id": id, "subtaskID": sid, "answer": ans})

subtasks = [
    ([1], 1, subtask1),
    ([2], 2, subtask2),
    ([3], 3, subtask3),
    ([4], 4, subtask4),
    (subtask5.index, 5, subtask5.values),
    (muzee["_id"], 6, subtask6),
    ([7], 7, subtask7),
    ([8], 8, subtask8),
]

submission = pd.concat([build_subtask(id, sid, ans) for id, sid, ans in subtasks])

In [ ]:
submission.head()

In [ ]:
submission.to_csv(f"{root_path}/submission.csv", index=False)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)